<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>

<br>

# <font color="#76b900">**Notebook 9:** LangServe and Assessment</font>

<br>

## LangServe Server Setup

This notebook is a playground for those interested in developing interactive web applications using LangChain and [**LangServe**](https://python.langchain.com/docs/langserve). The aim is to provide a minimal-code example to illustrate the potential of LangChain in web application contexts.

This section provides a walkthrough for setting up a simple API server using LangChain's Runnable interfaces with FastAPI. The example demonstrates how to integrate a LangChain model, such as `ChatNVIDIA`, to create and distribute accessible API routes. Using this, you will be able to supply functionality to the frontend service's [**`frontend_server.py`**](frontend/frontend_server.py) session, which strongly expects:
- A simple endpoint named `:9012/basic_chat` for the basic chatbot, exemplified below.
- A pair of endpoints named `:9012/retriever` and `:9012/generator` for the RAG chatbot.
- All three for the **Evaluate** utility, which will be required for the final assessment. *More on that later!*

**IMPORTANT NOTES:**
- Make sure to click the square ( $\square$ ) button twice to shut down an active FastAPI cell. The first time might fall through or trigger a try-catch routine on an asynchronous process.
- If it still doesn't work, do a hard restart on this notebook by using **Kernel -> Restart Kernel**.
- When a FastAPI server is running in your cell, expect the process to block up this notebook. Other notebooks should not be impacted by this. 

<br>

### **Part 1:** Delivering the /basic_chat endpoint

Instructions are provided for launching a `/basic_chat` endpoint both as a standalone Python file. This will be used by the frontend to make basic decision with no internal reasoning.

In [ ]:
# %%writefile server_app.py
# https://python.langchain.com/docs/langserve#server
from fastapi import FastAPI
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langserve import add_routes

## May be useful later
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.prompt_values import ChatPromptValue
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_community.document_transformers import LongContextReorder
from functools import partial
from operator import itemgetter

from langchain_community.vectorstores import FAISS

## TODO: Make sure to pick your LLM and do your prompt engineering as necessary for the final assessment
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")
instruct_llm = ChatNVIDIA(model="meta/llama3-8b-instruct")

app = FastAPI(
  title="LangChain Server",
  version="1.0",
  description="A simple api server using Langchain's Runnable interfaces",
)

## PRE-ASSESSMENT: Run as-is and see the basic chain in action

add_routes(
    app,
    instruct_llm,
    path="/basic_chat",
)

## ASSESSMENT TODO: Implement these components as appropriate

add_routes(
    app,
    RunnableLambda(lambda x: "Not Implemented"),
    path="/generator",
)

add_routes(
    app,
    RunnableLambda(lambda x: []),
    path="/retriever",
)

## Might be encountered if this were for a standalone python file...
# if __name__ == "__main__":
#     import uvicorn
#     uvicorn.run(app, host="0.0.0.0", port=9012)

In [4]:
## Works, but will block the notebook.
!python server_app.py  

## Will technically work, but not recommended in a notebook. 
## You may be surprised at the interesting side effects...
# import os
# os.system("python server_app.py &")

^C


In [1]:
print("hello")

hello


<br>

### **Part 2:** Using The Server:

While this cannot be easily utilized within Google Colab (or at least not without a lot of special tricks), the above script will keep a running server tied to the notebook process. While the server is running, do not attempt to use this notebook (except to shut down/restart the service).

In another file, however, you should be able to access the `basic_chat` endpoint using the following interface:

```python
from langserve import RemoteRunnable
from langchain_core.output_parsers import StrOutputParser

llm = RemoteRunnable("http://0.0.0.0:9012/basic_chat/") | StrOutputParser()
for token in llm.stream("Hello World! How is it going?"):
    print(token, end='')
```

**Please try it out in a different file and see if it works!**


<br>

### **Part 3: Final Assessment**

**This notebook will be used to completing the final assessment!** When you have otherwise finished the course, we recommend cloning this notebook, getting the frontend open in a new tab, and implement the Evaluate functionality by implementing the `/generator` and `/retriever` endpoints above! For a quick link to the frontend, run the cell below:

In [2]:
%%js
var url = 'http://'+window.location.host+':8090';
element.innerHTML = '<a style="color:#76b900;" target="_blank" href='+url+'><h2>< Link To Gradio Frontend ></h2></a>';

<IPython.core.display.Javascript object>

<hr>
<br>

#### **Assessment Hint:** 
Note that the following functionality is already implemented in the frontend microservice. 

```python
## Necessary Endpoints
chains_dict = {
    'basic' : RemoteRunnable("http://lab:9012/basic_chat/"),
    'retriever' : RemoteRunnable("http://lab:9012/retriever/"),  ## For the final assessment
    'generator' : RemoteRunnable("http://lab:9012/generator/"),  ## For the final assessment
}

basic_chain = chains_dict['basic']

## Retrieval-Augmented Generation Chain

retrieval_chain = (
    {'input' : (lambda x: x)}
    | RunnableAssign(
        {'context' : itemgetter('input') 
        | chains_dict['retriever'] 
        | LongContextReorder().transform_documents
        | docs2str
    })
)

output_chain = RunnableAssign({"output" : chains_dict['generator'] }) | output_puller
rag_chain = retrieval_chain | output_chain
```

**To conform to this endpoint ingestion strategy, make sure not to duplicate pipeline functionality and only deploy the features that are missing!**

----

<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>

In [ ]:
"""
Advanced RAG Chat: documents + chat history retrieval + Gradio UI
Requirements (install):
  pip install langchain langchain-nvidia-ai-endpoints langchain-community faiss-cpu gradio sklearn
Adjust imports if you use different langchain package names in your env.
"""
import json
from pprint import pprint
from functools import partial
import re

# langchain / NVIDIA imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import ArxivLoader  # or replace with local loader(s)
from langchain_community.vectorstores import FAISS
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA

# for building HNSW FAISS index (ANN)
from faiss import IndexHNSWFlat
from langchain_community.docstore.in_memory import InMemoryDocstore

# simple numeric tools
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Gradio UI
import gradio as gr

# ---------------------------------------------------------
# 1) Prepare embedder + model (NVIDIA)
# ---------------------------------------------------------
# Choose the NVIDIA embedding + chat models available to you.
# Adjust the model names to what's available in your environment.
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")
chat_model = ChatNVIDIA(model="mistralai/mixtral-8x7b-instruct-v0.1")

# ---------------------------------------------------------
# 2) Load & chunk documents (your text_splitter)
# ---------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100,
    separators=["\n\n", "\n", ".", ";", ",", " "],
)

print("Loading sample papers...")
docs_raw = [
    ArxivLoader(query="1706.03762").load(),  # Attention is All You Need
    ArxivLoader(query="1810.04805").load(),  # BERT
    ArxivLoader(query="2005.11401").load(),  # RAG
]

# Flatten loader outputs and remove trailing References sections
all_papers = []
for doclist in docs_raw:
    for d in doclist:
        # truncate at "References" if present
        content = d.page_content
        idx = content.find("References")
        if idx != -1:
            d.page_content = content[:idx]
        all_papers.append(d)

# chunk documents
print("Chunking documents...")
docs_chunks = text_splitter.split_documents(all_papers)
# filter out tiny chunks
docs_chunks = [c for c in docs_chunks if len(c.page_content.strip()) > 200]

# keep a parallel list of raw page_content for quick keyword search
all_chunk_texts = [c.page_content for c in docs_chunks]

# ---------------------------------------------------------
# 3) Build a persistent-ish ANN FAISS docstore (HNSW)
# ---------------------------------------------------------
# Build an HNSW index, wrapped by LangChain FAISS wrapper.
# IndexHNSWFlat requires specifying embedding dim: compute by embedding a test query.
embed_dim = len(embedder.embed_query("test"))  # may perform network call

def make_empty_hnsw_faiss():
    index = IndexHNSWFlat(embed_dim, 32)  # M=32 (neighbors param) — tune for your workload
    return FAISS(
        embedding_function=embedder,
        index=index,
        docstore=InMemoryDocstore(),   # for persistence use a backed docstore instead
        index_to_docstore_id={},
        normalize_L2=True
    )

# Build FAISS from chunks (LangChain will create an index for you, but we'll use from_documents to keep simple)
print("Creating FAISS docstore for document chunks (this will embed chunks)...")
docstore = FAISS.from_documents(docs_chunks, embedding=embedder)  # builds default index (may be flat)
# Optionally: if you want HNSW specifically, you could create empty HNSW and merge_from other stores:
# hnsw_store = make_empty_hnsw_faiss()
# hnsw_store.merge_from(docstore); docstore = hnsw_store

# ---------------------------------------------------------
# 4) Conversation memory FAISS (sliding-window memory)
# ---------------------------------------------------------
print("Initializing conversation memory store...")
# Create an initially empty convstore using HNSW to support fast incremental adds
convstore = make_empty_hnsw_faiss()
# We don't add initial entries; convstore.add_texts(...) will append as the session runs.

# We'll also maintain an in-memory list for the chat UI's display (gradio uses a separate state)
conversation_history_list = []  # list[str] pairs [user, agent, user, agent, ...]

# ---------------------------------------------------------
# 5) Retrieval helpers: semantic, keyword, hybrid, rerank
# ---------------------------------------------------------
def semantic_retrieval(query: str, k: int = 5):
    """Use FAISS retriever to get top-k documents semantically similar to query."""
    retriever = docstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.get_relevant_documents(query)
    return docs

def keyword_retrieval(query: str, k: int = 5):
    """Simple keyword search: returns chunks containing any query tokens.
       This is a cheap, first-pass keyword filter (replace with BM25 for production).
    """
    tokens = [t for t in re.split(r"\W+", query) if len(t) > 2]
    results = []
    for i, txt in enumerate(all_chunk_texts):
        score = sum(1 for t in tokens if re.search(r"\b" + re.escape(t) + r"\b", txt, flags=re.I))
        if score > 0:
            results.append((score, i))
    # sort by score desc and return top k Document objects from docs_chunks
    results = sorted(results, key=lambda x: x[0], reverse=True)[:k]
    return [docs_chunks[i] for _, i in results]

def hybrid_retrieval(query: str, k:int = 8):
    """Combine semantic + keyword retrieval and de-duplicate."""
    sem = semantic_retrieval(query, k=k)
    kw = keyword_retrieval(query, k=k)
    # combine by textual content uniqueness
    seen = set()
    combined = []
    for d in sem + kw:
        key = d.page_content[:200]  # small fingerprint
        if key not in seen:
            combined.append(d)
            seen.add(key)
        if len(combined) >= k:
            break
    return combined

def rerank_by_embedding_similarity(query: str, docs, top_n=5):
    """Simple re-ranker: compute query embedding and cosine similarity to doc embeddings.
       This is not a heavy cross-encoder but is effective after ANN retrieval.
    """
    if not docs:
        return []
    # embed query and docs
    q_emb = np.array(embedder.embed_query(query)).reshape(1, -1)
    doc_texts = [d.page_content for d in docs]
    doc_embs = np.array(embedder.embed_documents(doc_texts))
    sims = cosine_similarity(q_emb, doc_embs)[0]  # shape (n,)
    idxs = np.argsort(-sims)[:top_n]
    return [docs[i] for i in idxs]

# ---------------------------------------------------------
# 6) Formatting utilities for LLM prompt
# ---------------------------------------------------------
def docs2context_string(docs, max_chars_per_doc=1000):
    """Format a list of Document objects into one textual context for the LLM"""
    out = []
    for i, d in enumerate(docs):
        # include a short source tag if metadata exists
        meta = getattr(d, "metadata", {}) or {}
        title = meta.get("Title") or meta.get("source") or f"doc_{i}"
        excerpt = d.page_content[:max_chars_per_doc].strip()
        out.append(f"[Source: {title}]\n{excerpt}")
    return "\n\n".join(out)

def history2context_string(history_list, keep_last_n_turns=6):
    """Turn recent conversation turns into context for the LLM"""
    # history_list is like: [user1, agent1, user2, agent2, ...]
    # we want the last keep_last_n_turns turns (each turn has user+agent)
    text = []
    # keep last N pairs (user,agent)
    pairs = []
    for i in range(0, len(history_list), 2):
        user = history_list[i] if i < len(history_list) else ""
        agent = history_list[i+1] if i+1 < len(history_list) else ""
        pairs.append((user, agent))
    recent = pairs[-keep_last_n_turns:]
    for u,a in recent:
        text.append(f"[User] {u}")
        if a:
            text.append(f"[Agent] {a}")
    return "\n".join(text)

# ---------------------------------------------------------
# 7) Saving new memory to convstore (conversation FAISS)
# ---------------------------------------------------------
def save_to_convstore(user_text: str, agent_text: str):
    """Add user/agent lines to the memory FAISS store and to the in-memory history list."""
    if user_text:
        convstore.add_texts([f"[User] {user_text}"], metadatas=[{"role":"user"}])
        conversation_history_list.append(user_text)
    if agent_text:
        convstore.add_texts([f"[Agent] {agent_text}"], metadatas=[{"role":"agent"}])
        conversation_history_list.append(agent_text)

# ---------------------------------------------------------
# 8) Build prompt and call LLM
# ---------------------------------------------------------
from langchain.prompts import ChatPromptTemplate

system_template = (
    "You are a knowledgeable assistant that answers user questions using only the provided context. "
    "Cite sources in-line like [Source: title]. Answer conversationally."
)

# Prompt template: we will insert three pieces: recent history, retrieved documents, and user question
prompt_template = ChatPromptTemplate.from_template(
    system_template + "\n\n"
    "Conversation history:\n{history}\n\n"
    "Retrieved documents (most relevant first):\n{context}\n\n"
    "User question:\n{question}\n\n"
    "Answer below conversationally and only using the retrieved context."
)

def call_llm_with_prompt(history_text: str, retrieved_text: str, question: str) -> str:
    """Construct the prompt and call the NVIDIA chat model.
       Replace the call below with the exact invocation your ChatNVIDIA wrapper expects.
    """
    prompt = prompt_template.format_prompt(history=history_text, context=retrieved_text, question=question).to_string()
    # --------------------
    # Option A: if ChatNVIDIA supports a simple text-in/text-out call:
    # response = chat_model.call_as_text(prompt)   # <-- example placeholder
    # --------------------
    # Option B: use a Chat completion style if supported by the SDK:
    # messages = [{"role":"system", "content": system_template},
    #             {"role":"user", "content": prompt}]
    # response = chat_model.chat(messages)
    # --------------------
    # I'll use a generic 'chat_model.generate' placeholder — replace with actual call:
    try:
        # Many wrappers accept a simple call like: chat_model.generate([prompt]) or chat_model(prompt)
        response = chat_model(prompt)  # attempt __call__ ; adapt if your object uses `.generate()` or `.chat()`
    except Exception as e:
        # Fallback: if the wrapper doesn't support __call__, use a pseudo-return for now
        response = "ERROR_CALLING_CHAT_MODEL: replace this call with the correct ChatNVIDIA invocation. " \
                   f"Exception: {e}"
    # If response is an object with .text or .generations, extract the text appropriately:
    if hasattr(response, "text"):
        return response.text
    if isinstance(response, list) and response and isinstance(response[0], str):
        return response[0]
    return str(response)

# ---------------------------------------------------------
# 9) Orchestrator: given user question -> retrieve -> answer -> save memory
# ---------------------------------------------------------
def handle_user_query(user_question: str, k_semantic=6, k_rerank=4):
    # 1. Build recent convo history text (sliding window)
    history_text = history2context_string(conversation_history_list, keep_last_n_turns=3)

    # 2. Hybrid retrieval
    candidates = hybrid_retrieval(user_question, k=k_semantic)

    # 3. Rerank top-k (cheap embedding-based re-rank)
    reranked = rerank_by_embedding_similarity(user_question, candidates, top_n=k_rerank)

    # 4. Format retrieved docs
    retrieved_text = docs2context_string(reranked, max_chars_per_doc=800)

    # 5. Call LLM
    answer = call_llm_with_prompt(history_text, retrieved_text, user_question)

    # 6. Save to convo memory (so future queries can retrieve this turn)
    save_to_convstore(user_question, answer)

    return answer, retrieved_text, history_text

# ---------------------------------------------------------
# 10) Gradio UI wiring
# ---------------------------------------------------------
def gradio_chat(user_message, chat_history):
    """
    Gradio expects: inputs (message, chat_history)
    - chat_history is a list of pairs [[user, bot], ...]
    """
    # Convert chat_history (pairs) into our conversation_history_list if starting fresh
    # We'll not reinitialize convstore here — assume it's persisted in memory for this session.

    # Call the orchestrator
    reply, retrieved, history = handle_user_query(user_message)

    # Update gradio chat_history (append new pair)
    chat_history = chat_history or []
    chat_history.append((user_message, reply))
    return chat_history, retrieved  # second output can be a debug panel

with gr.Blocks() as demo:
    gr.Markdown("# Document + Chat Memory RAG Demo")
    chatbot = gr.Chatbot()
    state = gr.State([])  # chat history list of pairs
    debug = gr.Textbox(label="Retrieved context (debug)", lines=10)

    msg = gr.Textbox(placeholder="Ask something about the loaded papers or your prior chat...", label="Your message")
    submit = gr.Button("Send")

    # on submit, call gradio_chat and update UI
    submit.click(fn=gradio_chat, inputs=[msg, state], outputs=[chatbot, debug])

# Launch (local)
demo.launch(server_name="0.0.0.0", server_port=7860, share=False)


In [ ]:
# Full LCEL professional RAG chat with NVIDIA embeddings, HNSW FAISS, reranking, caching, sliding-window memory, and Gradio UI.

import re
import time
from functools import partial
from pprint import pprint

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

import gradio as gr

# LangChain core / runnables / prompts
from langchain_core.runnables import RunnableLambda, RunnableAssign
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# NVIDIA integration
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
# optional reranker - may not exist in all environments
try:
    from langchain_nvidia_ai_endpoints import NVIDIARerank
    HAS_NVIDIA_RERANK = True
except Exception:
    HAS_NVIDIA_RERANK = False

# FAISS / docstore
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from faiss import IndexHNSWFlat

# Helpers for chunking / loading (user already has text_splitter + ArxivLoader, adapt as needed)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import ArxivLoader

# -------------------------
# 0) Config - tune these for production
# -------------------------
HNSW_M = 32               # HNSW connectivity (trade memory <-> speed)
HNSW_EF_SEARCH = 64       # efSearch param for HNSW (set on search if API supports)
SLIDING_WINDOW_TURNS = 6  # keep last N user-agent turns for immediate context
TOP_K_RETRIEVE = 8
TOP_K_RERANK = 5
CACHE_MAX_ITEMS = 2000    # keep a capped cache for frequent queries

# -------------------------
# 1) Initialize NVIDIA embedder + model + optional reranker
# -------------------------
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")
# Chat model (small variant here; choose larger if available)
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x7b-instruct-v0.1")

if HAS_NVIDIA_RERANK:
    try:
        reranker = NVIDIARerank(model="nemo-retriever-reranker")
    except Exception:
        reranker = None
else:
    reranker = None

# compute embedding dim for FAISS HNSW
embed_dim = len(embedder.embed_query("test"))

# -------------------------
# 2) Build (or load) document chunks and docstore
# -------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100,
    separators=["\n\n", "\n", ".", ";", ",", " "],
)

# Example: load a few Arxiv documents ( adapt / replace with your list )
raw_papers = [
    ArxivLoader(query="1706.03762").load(),  # Attention Is All You Need
    ArxivLoader(query="1810.04805").load(),  # BERT
    ArxivLoader(query="2005.11401").load(),  # RAG
]

# Flatten and truncate at "References" (safe preprocessing)
all_docs = []
for loader_output in raw_papers:
    for doc in loader_output:
        content = doc.page_content
        idx = content.find("References")
        if idx != -1:
            doc.page_content = content[:idx]
        all_docs.append(doc)

# chunk documents
all_chunks = text_splitter.split_documents(all_docs)
all_chunks = [c for c in all_chunks if len(c.page_content.strip()) > 200]

# Build FAISS docstore from chunks (this will embed chunks)
print("Embedding & indexing document chunks (this may take time)...")
docstore = FAISS.from_documents(all_chunks, embedding=embedder)

# optionally convert to HNSW for speed: create empty HNSW index then merge
def make_empty_hnsw_faiss():
    idx = IndexHNSWFlat(embed_dim, HNSW_M)
    return FAISS(
        embedding_function=embedder,
        index=idx,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
        normalize_L2=True
    )

# If user wants HNSW specifically, merge existing docstore into an HNSW wrapper:
hnsw_store = make_empty_hnsw_faiss()
hnsw_store.merge_from(docstore)
docstore = hnsw_store
print("Docstore size:", len(docstore.docstore._dict))

# -------------------------
# 3) Conversation memory store (FAISS HNSW) - dynamic updates
# -------------------------
convstore = make_empty_hnsw_faiss()

# in-memory human-friendly list used for the UI history; convstore holds the searchable records
conversation_history_list = []  # stores alternating user/agent strings

# -------------------------
# 4) Simple LRU-ish cache for embeddings and query results
# -------------------------
class SimpleCache:
    def __init__(self, max_items=1000):
        self.max = max_items
        self._store = {}
        self._order = []

    def get(self, key):
        v = self._store.get(key)
        if v is not None:
            # bump recency
            try:
                self._order.remove(key)
            except ValueError:
                pass
            self._order.append(key)
        return v

    def set(self, key, value):
        if key in self._store:
            try:
                self._order.remove(key)
            except ValueError:
                pass
        self._store[key] = value
        self._order.append(key)
        if len(self._order) > self.max:
            oldest = self._order.pop(0)
            del self._store[oldest]

# caches
embedding_cache = SimpleCache(max_items=CACHE_MAX_ITEMS)
retrieval_cache = SimpleCache(max_items=CACHE_MAX_ITEMS)

def cached_embed_query(q):
    v = embedding_cache.get(q)
    if v is not None:
        return v
    emb = embedder.embed_query(q)
    embedding_cache.set(q, emb)
    return emb

# Precompute embeddings for titles/metadata (metadata-aware cache)
metadata_items = []
for doc_id, doc in enumerate(all_chunks[:200]):  # sample metadata precompute
    meta = getattr(doc, "metadata", {}) or {}
    title = meta.get("Title") or meta.get("source") or f"doc_{doc_id}"
    metadata_items.append(title)
for m in metadata_items:
    cached_embed_query(m)

# -------------------------
# 5) Retrieval functions (semantic, keyword, hybrid, rerank)
# -------------------------
def semantic_retrieve_cached(query, k=TOP_K_RETRIEVE):
    """Use docstore retriever with optional caching of results (by query string)."""
    cache_key = f"sem:{query}:{k}"
    cached = retrieval_cache.get(cache_key)
    if cached is not None:
        return cached
    # get retriever and optionally set efSearch if available via search_kwargs
    retr = docstore.as_retriever(search_kwargs={"k": k})
    docs = retr.get_relevant_documents(query)
    retrieval_cache.set(cache_key, docs)
    return docs

def keyword_retrieve(query, k=TOP_K_RETRIEVE):
    tokens = [t for t in re.split(r"\W+", query) if len(t) > 2]
    results = []
    for i, chunk in enumerate(all_chunks):
        txt = chunk.page_content
        score = sum(1 for t in tokens if re.search(r"\b" + re.escape(t) + r"\b", txt, flags=re.I))
        if score > 0:
            results.append((score, i))
    results = sorted(results, key=lambda x: x[0], reverse=True)[:k]
    return [all_chunks[i] for _, i in results]

def hybrid_retrieve(query, k=TOP_K_RETRIEVE):
    # Combine semantic + keyword, de-duplicate
    sem = semantic_retrieve_cached(query, k=k)
    kw = keyword_retrieve(query, k=k)
    seen = set()
    combined = []
    for d in sem + kw:
        fingerprint = (d.metadata.get("source") if d.metadata else None, d.page_content[:200])
        if fingerprint not in seen:
            combined.append(d)
            seen.add(fingerprint)
        if len(combined) >= k:
            break
    return combined

def rerank_candidates(query, candidates, top_n=TOP_K_RERANK):
    if not candidates:
        return []
    # If NVIDIA re-ranker is available, call it
    if reranker is not None:
        try:
            # expected API: reranker.rerank(query, [doc_texts]) -> returns sorted docs; adapt if different
            doc_texts = [d.page_content for d in candidates]
            ranked_indices = reranker.rerank(query, doc_texts)  # pseudo-call; adapt if SDK differs
            ranked = [candidates[i] for i in ranked_indices][:top_n]
            return ranked
        except Exception:
            pass
    # fallback: embedding-based cosine similarity re-rank
    q_emb = np.array(cached_embed_query(query)).reshape(1, -1)
    doc_texts = [d.page_content for d in candidates]
    doc_embs = np.array(embedder.embed_documents(doc_texts))
    sims = cosine_similarity(q_emb, doc_embs)[0]
    idxs = np.argsort(-sims)[:top_n]
    return [candidates[i] for i in idxs]

# -------------------------
# 6) Format helpers
# -------------------------
def docs2str(docs, max_chars=1000):
    out = []
    for i, d in enumerate(docs):
        meta = getattr(d, "metadata", {}) or {}
        title = meta.get("Title") or meta.get("source") or f"doc_{i}"
        excerpt = d.page_content[:max_chars].strip()
        out.append(f"[Source: {title}]\n{excerpt}")
    return "\n\n".join(out)

def history2str(history_list, last_n_pairs=SLIDING_WINDOW_TURNS):
    # history_list = [user1, agent1, user2, agent2, ...]
    pairs = []
    for i in range(0, len(history_list), 2):
        u = history_list[i] if i < len(history_list) else ""
        a = history_list[i+1] if i+1 < len(history_list) else ""
        pairs.append((u, a))
    recent = pairs[-last_n_pairs:]
    out = []
    for u,a in recent:
        out.append(f"[User] {u}")
        if a:
            out.append(f"[Agent] {a}")
    return "\n".join(out)

# -------------------------
# 7) Save memory runnable (dynamic updates)
# -------------------------
def save_memory_and_return_output(d, vstore=convstore):
    """d is a dict with keys 'input' and 'output' (the LLM text)."""
    user_text = d.get("input", "")
    agent_text = d.get("output", "")
    if user_text:
        vstore.add_texts([f"[User] {user_text}"], metadatas=[{"role":"user"}])
        conversation_history_list.append(user_text)
    if agent_text:
        vstore.add_texts([f"[Agent] {agent_text}"], metadatas=[{"role":"agent"}])
        conversation_history_list.append(agent_text)
    # keep sliding memory list bounded
    if len(conversation_history_list) > SLIDING_WINDOW_TURNS * 2:
        # keep recent only
        conversation_history_list[:] = conversation_history_list[-SLIDING_WINDOW_TURNS*2:]
    return agent_text

# -------------------------
# 8) Prompt template
# -------------------------
system_text = (
    "You are a document-aware assistant. Use ONLY the retrieved context (history + documents) to answer. "
    "Cite sources inline as [Source: title]. If the answer is not contained in the retrieval, say 'I don't know.'"
)
prompt_template = ChatPromptTemplate.from_template(
    system_text + "\n\nConversation history:\n{history}\n\nRetrieved documents:\n{context}\n\nUser question:\n{input}\n\nAnswer:"
)

# -------------------------
# 9) LCEL runnables (retrieval chain + prompt + llm + memory-save)
# -------------------------

# A. Base identity for input
identity = RunnableLambda(lambda x: x)

# B. Retrieval runnable: build history string and retrieve/rerank docs -> produce 'history' & 'context'
def retrieval_assignments(d):
    q = d["input"]
    # 1) history from convstore (sliding window) - use convstore retriever to get past related turns
    history_docs = convstore.as_retriever(search_kwargs={"k": 6}).get_relevant_documents(q)
    # format history: include both convstore hits and the in-memory list for recency
    recent_history_str = history2str(conversation_history_list, last_n_pairs=SLIDING_WINDOW_TURNS)
    history_from_index = docs2str(history_docs, max_chars=600)
    history_combined = recent_history_str + ("\n\n" + history_from_index if history_from_index else "")

    # 2) hybrid doc retrieval + rerank
    candidates = hybrid_retrieve(q, k=TOP_K_RETRIEVE)
    reranked = rerank_candidates(q, candidates, top_n=TOP_K_RERANK)
    context_str = docs2str(reranked, max_chars=900)
    return {"history": history_combined, "context": context_str}

retrieval_runnable = RunnableAssign({"history": lambda d: retrieval_assignments(d)["history"],
                                     "context": lambda d: retrieval_assignments(d)["context"]})

# C. Compose full chain: input -> retrieval -> prompt -> llm -> parse -> save memory
conv_chain = (
    {"input": identity}   # start: map incoming message to 'input'
    | retrieval_runnable  # populates 'history' and 'context' keys
    | RunnableAssign({"output": prompt_template | instruct_llm | StrOutputParser()})
    | RunnableLambda(lambda d: save_memory_and_return_output(d))
)

# -------------------------
# 10) Gradio UI wrapper using conv_chain.invoke
# -------------------------
def gradio_step(user_message, chat_history):
    # chat_history is a list of tuples (user, bot) used only for display
    if not user_message:
        return chat_history, ""
    # call the LCEL chain
    result = conv_chain.invoke(user_message)
    # conv_chain returns the string output from save_memory_and_return_output (the agent text)
    agent_text = result
    # append to gradio chat_history and return debug retrieval snippet
    chat_history = chat_history or []
    chat_history.append((user_message, agent_text))
    # For debug, also show last retrieval results (we can reuse the retrieval functions to get last context)
    # (Note: this duplicates work; for production move retrieval debug into chain output)
    last_candidates = hybrid_retrieve(user_message, k=TOP_K_RETRIEVE)
    last_reranked = rerank_candidates(user_message, last_candidates, top_n=TOP_K_RERANK)
    debug_context = docs2str(last_reranked, max_chars=800)
    return chat_history, debug_context

# Launch Gradio
with gr.Blocks() as demo:
    gr.Markdown("# Pro RAG Chat (LCEL) — Documents + Chat Memory")
    chatbot = gr.Chatbot()
    state = gr.State([])  # gradio internal state storing chat_history pairs
    debug = gr.Textbox(label="Retrieved context (debug)", lines=10)
    msg = gr.Textbox(placeholder="Ask anything about the documents or your chat history...")
    send = gr.Button("Send")

    send.click(fn=gradio_step, inputs=[msg, state], outputs=[chatbot, debug])
    # allow pressing enter to send
    msg.submit(fn=gradio_step, inputs=[msg, state], outputs=[chatbot, debug])

print("Launching Gradio UI...")
demo.launch(server_name="0.0.0.0", server_port=7860, share=False)
